# Notebook 4: Evaluation

## 4.0 Preamble

This notebook benchmarks all trained driving policies -- the behavior-cloning (BC) baseline and the three PPO driving-style variants (chill, standard, hurry) -- across a standardised evaluation grid of towns, weather conditions, and episodes. The goal is to quantify how much PPO fine-tuning improves over pure imitation learning, whether those improvements generalize across unseen towns and adverse weather, and how the three driving styles differ in practice. Statistical significance tests ensure that observed differences are not due to random variation in episode outcomes.

In [ ]:
import sys
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Ensure project root is on sys.path so src.* imports resolve
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.agents.eval_agent import EvaluationAgent

print(f"Project root : {PROJECT_ROOT}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")

## 4.1 Configuration

### 4.1.1 Loading Parameters

All evaluation parameters are stored in `configs/eval.yaml` and loaded via `load_config("eval")`. This includes the set of weather presets to test, the number of episodes per condition, and the maximum steps per episode. Loading from YAML ensures that the notebook and the EvaluationAgent always operate with identical settings. The crop constants for both BC and PPO models are also specified here to avoid hard-coded magic numbers.

In [ ]:
cfg = load_config("eval")

print("=== Evaluation Parameters ===")
print(f"  eval_towns             : {cfg['eval_towns']}")
print(f"  eval_weathers          : {cfg['eval_weathers']}")
print(f"  episodes_per_condition : {cfg['episodes_per_condition']}")
print(f"  max_steps_per_episode  : {cfg['max_steps_per_episode']}")
print(f"  sensor_suites          : {cfg['sensor_suites']}")
print(f"  grp_sampling           : {cfg['grp_sampling']}")
print(f"  crop (PPO)             : [{cfg['crop']['ppo_crop_top']}:{cfg['crop']['ppo_crop_bottom']}]")
print(f"  crop (BC)              : [{cfg['crop']['bc_crop_top']}:{cfg['crop']['bc_crop_bottom']}]")

### 4.1.2 Evaluation Grid

The evaluation grid tests every combination of sensor suite, model, weather, town, and episode. The EvaluationAgent runs all three eval towns (Town01, Town03, Town05) and all weather conditions autonomously for a given sensor suite. Run NB04 once per sensor suite.

| Dimension | Values | Count |
|---|---|---|
| Sensor suites | single_cam, multi_cam, lidar | 3 |
| Models per suite | BC, PPO-chill, PPO-standard, PPO-hurry | 4 |
| Weathers | ClearNoon, HardRainNoon, ClearNight | 3 |
| Towns | Town01, Town03, Town05 | 3 |
| Episodes per condition | 10 | 10 |
| **Total** | | **1,080** |

## 4.2 Running Evaluation

### 4.2.1 Autonomous Multi-Town Evaluation

The EvaluationAgent manages the full CARLA lifecycle automatically for each eval town. You do not need to start or stop CARLA manually. Run each cell below for one sensor suite. Each run takes approximately 3-4 hours on an RTX 5080.

Run this notebook three times (once per sensor suite) to collect data for all three model families. Results from each run are appended to `results/eval_results.json` with deduplication by sensor suite.

In [ ]:
# Single-camera baseline (control condition for causal analysis)
# Do NOT start CARLA; the agent manages it automatically.
agent_single = EvaluationAgent(
    sensor_suite="single_cam",
    record_video=True,
)
records_single = agent_single.run()
print(f"Single-cam evaluation complete: {len(records_single)} episodes.")

### 4.2.2 Run Multi-Camera and Lidar Evaluations

Run these cells after the single-cam cell above is complete.

In [ ]:
# Multi-camera (Tesla HydraNet-style late fusion)
agent_multi = EvaluationAgent(
    sensor_suite="multi_cam",
    record_video=True,
)
records_multi = agent_multi.run()
print(f"Multi-cam evaluation complete: {len(records_multi)} episodes.")

In [ ]:
# Lidar BEV projection (Waymo-style)
agent_lidar = EvaluationAgent(
    sensor_suite="lidar",
    record_video=True,
)
records_lidar = agent_lidar.run()
print(f"Lidar evaluation complete: {len(records_lidar)} episodes.")

## 4.3 Route Completion Analysis

### 4.3.1 Behavior Cloning vs Proximal Policy Optimization

This section answers the central question: does PPO fine-tuning improve route completion over the BC baseline? Route completion is the fraction of the planned route that the agent successfully traverses before a collision, timeout, or reaching the destination. A higher value means the agent drives farther along its intended path. We aggregate across all towns, weathers, and episodes to get a robust estimate of each model type's overall capability.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_path = results_dir / "eval_results.json"

with open(results_path) as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} evaluation records from {results_path.name}")
print(f"Sensor suites: {sorted(df['sensor_suite'].unique())}")

# Mean route completion by sensor suite (primary causal treatment)
rc_by_suite = df.groupby("sensor_suite")["route_completion"].agg(["mean", "std", "count"])
print("\n=== Route Completion by Sensor Suite ===")
print(rc_by_suite.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
suites = ["single_cam", "multi_cam", "lidar"]
suite_colors = {"single_cam": "#1976D2", "multi_cam": "#43A047", "lidar": "#E53935"}
for suite in suites:
    if suite in rc_by_suite.index:
        ax.bar(suite, rc_by_suite.loc[suite, "mean"],
               yerr=rc_by_suite.loc[suite, "std"],
               color=suite_colors[suite], edgecolor="black", capsize=5)
ax.set_ylabel("Mean Route Completion")
ax.set_title("Route Completion by Sensor Suite")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### 4.3.2 Cross-Town Generalization

This analysis answers whether PPO's improvement over BC is consistent across different towns, or if it only works well in certain environments. Town01 is a simple suburban layout, Town03 has more complex intersections and roundabouts, and Town05 is a multi-lane urban environment. Consistent improvement across all three towns suggests that PPO is learning generalizable driving skills rather than memorising town-specific routes.

In [ ]:
rc_by_type_town = df.groupby(["model_type", "town"])["route_completion"].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 6))
rc_by_type_town.plot(kind="bar", ax=ax, edgecolor="black", width=0.7)
ax.set_ylabel("Mean Route Completion")
ax.set_title("Route Completion by Model Type and Town")
ax.set_ylim(0, 1.05)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Town")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4.4 Day vs Night Performance Gap

### 4.4.1 BC Day vs Night

Night driving is substantially harder for vision-based models because the camera images are darker, lane markings are less visible, and headlights create glare. This section quantifies the day-vs-night performance gap for the BC baseline by comparing ClearNoon and ClearNight conditions. A large gap indicates that the BC training data may have been biased toward daytime conditions, causing the model to struggle when visual features change at night.

In [ ]:
bc_df = df[df["model_type"] == "bc"]

bc_day = bc_df[bc_df["weather"] == "ClearNoon"]["route_completion"]
bc_night = bc_df[bc_df["weather"] == "ClearNight"]["route_completion"]

print("=== BC: Day vs Night ===")
print(f"  ClearNoon  -- mean: {bc_day.mean():.4f}, std: {bc_day.std():.4f}, n={len(bc_day)}")
print(f"  ClearNight -- mean: {bc_night.mean():.4f}, std: {bc_night.std():.4f}, n={len(bc_night)}")
print(f"  Gap: {bc_day.mean() - bc_night.mean():.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["ClearNoon", "ClearNight"],
       [bc_day.mean(), bc_night.mean()],
       yerr=[bc_day.std(), bc_night.std()],
       color=["#FFC107", "#37474F"], edgecolor="black", capsize=5)
ax.set_ylabel("Mean Route Completion")
ax.set_title("BC Model: Day vs Night")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### 4.4.2 PPO Day vs Night

This section answers whether PPO fine-tuning closes the day-vs-night performance gap compared to BC. Because PPO's curriculum introduces night weather in Phase 2 (after step 50,000), the PPO models have explicitly practised driving in ClearNight conditions. If the gap is smaller for PPO than for BC, it confirms that the weather curriculum is an effective strategy for building robustness to lighting changes.

In [ ]:
ppo_df = df[df["model_type"] == "ppo"]

ppo_day = ppo_df[ppo_df["weather"] == "ClearNoon"]["route_completion"]
ppo_night = ppo_df[ppo_df["weather"] == "ClearNight"]["route_completion"]

print("=== PPO: Day vs Night ===")
print(f"  ClearNoon  -- mean: {ppo_day.mean():.4f}, std: {ppo_day.std():.4f}, n={len(ppo_day)}")
print(f"  ClearNight -- mean: {ppo_night.mean():.4f}, std: {ppo_night.std():.4f}, n={len(ppo_night)}")
print(f"  Gap: {ppo_day.mean() - ppo_night.mean():.4f}")

# Side-by-side comparison
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [bc_day.mean(), bc_night.mean()], width, label="BC",
       color="#1976D2", edgecolor="black")
ax.bar(x + width/2, [ppo_day.mean(), ppo_night.mean()], width, label="PPO",
       color="#43A047", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(["ClearNoon", "ClearNight"])
ax.set_ylabel("Mean Route Completion")
ax.set_title("Day vs Night: BC vs PPO")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4.5 Lane Change Analysis

### 4.5.1 Lane Change Frequency by Model and Condition

Lane keeping fraction measures the proportion of timesteps where the agent stays within its lane without invasion. A higher value indicates more disciplined lane-keeping. This metric is especially relevant for comparing driving styles: the chill style is trained with a high lane-change penalty (2.0) and should show higher lane-keeping fractions, while the hurry style (penalty 0.5) may sacrifice lane discipline for speed. We break this down by model and weather to identify conditions where lane discipline degrades.

In [ ]:
# Lane invasion frequency = 1 - lane_keeping_frac
df["lane_invasion_frac"] = 1.0 - df["lane_keeping_frac"]

invasion_by_model_weather = df.groupby(["model", "weather"])["lane_invasion_frac"].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 6))
invasion_by_model_weather.plot(kind="bar", ax=ax, edgecolor="black", width=0.7)
ax.set_ylabel("Mean Lane Invasion Fraction")
ax.set_title("Lane Invasion Frequency by Model and Weather")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(title="Weather")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### 4.5.2 Lane Change Decision Heatmap

This heatmap answers the question: under which combinations of model and weather is it better to stay in lane versus change lanes? A high lane-keeping fraction in a particular cell means the model rarely invades adjacent lanes under that weather condition. Models that maintain high lane-keeping across all weathers demonstrate robust lateral control, while models that show weather-dependent drops may be struggling with visibility-related lane detection under rain or night conditions.

In [ ]:
lk_pivot = df.groupby(["model", "weather"])["lane_keeping_frac"].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(lk_pivot.values, cmap="YlGn", aspect="auto", vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(lk_pivot.columns)))
ax.set_xticklabels(lk_pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(lk_pivot.index)))
ax.set_yticklabels(lk_pivot.index)
ax.set_title("Lane Keeping Fraction: Model x Weather")

# Annotate cells
for i in range(len(lk_pivot.index)):
    for j in range(len(lk_pivot.columns)):
        val = lk_pivot.values[i, j]
        color = "white" if val < 0.75 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=10)

fig.colorbar(im, ax=ax, label="Lane Keeping Fraction")
plt.tight_layout()
plt.show()

## 4.6 Speed Decision Analysis

### 4.6.1 Speed Distribution by Model and Condition

This analysis answers the question: under which conditions is it better to speed up versus hold steady? Average speed captures how aggressively the agent drives. The hurry-style PPO is expected to drive faster on average, while the chill style should be slower and smoother. Weather conditions also affect speed -- models may slow down in rain due to reduced visibility or because the BC training data contained slower driving in adverse conditions. The box plot reveals the full distribution, including outliers where the agent was stuck or racing.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

models_list = sorted(df["model"].unique())
weathers = cfg["eval_weathers"]

positions = []
labels = []
data = []
pos = 0
for model_name in models_list:
    for weather in weathers:
        subset = df[(df["model"] == model_name) & (df["weather"] == weather)]
        if len(subset) > 0:
            data.append(subset["avg_speed_kmh"].values)
            positions.append(pos)
            labels.append(f"{model_name}\n{weather}")
            pos += 1
    pos += 0.5  # gap between models

bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#90CAF9")
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Avg Speed (km/h)")
ax.set_title("Speed Distribution by Model and Weather")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4.7 Driving Style Comparison

### 4.7.1 Chill vs Standard vs Hurry -- Route Completion

This section directly compares the three PPO driving styles on route completion. All three styles start from the same BC weights and train for the same number of timesteps; the only difference is the reward shaping profile. The chill style may achieve lower route completion because it drives more cautiously and slowly, while the hurry style may complete more of the route by driving faster -- but at the risk of more collisions. The standard style should provide a middle ground.

In [ ]:
ppo_styles_df = df[df["model_type"] == "ppo"].copy()

if len(ppo_styles_df) > 0:
    rc_by_style = ppo_styles_df.groupby("driving_style")["route_completion"].agg(["mean", "std", "count"])
    print("=== Route Completion by Driving Style ===")
    print(rc_by_style.to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    style_colors = {"chill": "#2196F3", "standard": "#4CAF50", "hurry": "#F44336"}
    styles_present = [s for s in ["chill", "standard", "hurry"] if s in rc_by_style.index]
    colors = [style_colors[s] for s in styles_present]
    ax.bar(styles_present,
           [rc_by_style.loc[s, "mean"] for s in styles_present],
           yerr=[rc_by_style.loc[s, "std"] for s in styles_present],
           color=colors, edgecolor="black", capsize=5)
    ax.set_ylabel("Mean Route Completion")
    ax.set_title("Route Completion: Chill vs Standard vs Hurry")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()
else:
    print("No PPO records found -- skipping style comparison.")

### 4.7.2 Chill vs Standard vs Hurry -- Jerk Profile

The jerk profile is approximated here by the variability in average speed across episodes. A model with high speed variability across episodes tends to accelerate and brake more aggressively, producing a jerkier ride. The chill style, trained with a high jerk penalty (2.0), should show lower speed variability than the hurry style (jerk penalty 0.5). This is an indirect proxy for ride comfort -- true jerk measurement requires per-frame acceleration data that is not recorded during evaluation.

In [ ]:
if len(ppo_styles_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    style_order = [s for s in ["chill", "standard", "hurry"]
                   if s in ppo_styles_df["driving_style"].unique()]
    speed_data = [ppo_styles_df[ppo_styles_df["driving_style"] == s]["avg_speed_kmh"].values
                  for s in style_order]
    bp = ax.boxplot(speed_data, labels=style_order, patch_artist=True, widths=0.5)
    colors_list = [style_colors.get(s, "#999") for s in style_order]
    for patch, color in zip(bp["boxes"], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_ylabel("Avg Speed (km/h)")
    ax.set_title("Speed Variability as Jerk Proxy: Chill vs Standard vs Hurry")
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()
else:
    print("No PPO records found -- skipping jerk profile comparison.")

### 4.7.3 Chill vs Standard vs Hurry -- Lane Change Rate

Lane-keeping fraction directly reflects how the lane-change penalty weight shapes driving behavior. The chill style (lane_change_penalty=2.0) is expected to maintain the highest lane-keeping fraction, staying in its lane almost exclusively. The hurry style (penalty=0.5) should show more frequent lane invasions because it is less penalized for crossing lane boundaries while pursuing speed. This comparison validates that the reward shaping is producing the intended behavioral differences.

In [ ]:
if len(ppo_styles_df) > 0:
    lk_by_style = ppo_styles_df.groupby("driving_style")["lane_keeping_frac"].agg(["mean", "std"])
    print("=== Lane Keeping Fraction by Driving Style ===")
    print(lk_by_style.to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    styles_present = [s for s in ["chill", "standard", "hurry"] if s in lk_by_style.index]
    colors = [style_colors[s] for s in styles_present]
    ax.bar(styles_present,
           [lk_by_style.loc[s, "mean"] for s in styles_present],
           yerr=[lk_by_style.loc[s, "std"] for s in styles_present],
           color=colors, edgecolor="black", capsize=5)
    ax.set_ylabel("Mean Lane Keeping Fraction")
    ax.set_title("Lane Keeping: Chill vs Standard vs Hurry")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()
else:
    print("No PPO records found -- skipping lane change comparison.")

## 4.8 Statistical Significance

### 4.8.1 Mann-Whitney U Results

Visual differences in bar charts can be misleading when sample sizes are small. The Mann-Whitney U test is a non-parametric test that determines whether two independent samples come from the same distribution. We apply it to all pairwise model comparisons on route completion. A p-value below 0.05 indicates a statistically significant difference. The Mann-Whitney U test is preferred over a t-test because route completion values are bounded between 0 and 1 and may not be normally distributed.

In [ ]:
model_types = df["model"].unique()
pairs = []
for i, m1 in enumerate(model_types):
    for m2 in model_types[i+1:]:
        rc1 = df[df["model"] == m1]["route_completion"].values
        rc2 = df[df["model"] == m2]["route_completion"].values
        if len(rc1) >= 2 and len(rc2) >= 2:
            stat, p = stats.mannwhitneyu(rc1, rc2, alternative="two-sided")
            pairs.append({
                "Model A": m1,
                "Model B": m2,
                "U statistic": round(stat, 2),
                "p-value": round(p, 6),
                "Significant (p<0.05)": "Yes" if p < 0.05 else "No",
            })

df_mw = pd.DataFrame(pairs)
print("=== Pairwise Mann-Whitney U Tests (route_completion) ===")
print(df_mw.to_string(index=False))

### 4.8.2 Effect Size

Statistical significance alone does not tell us how large the difference is -- a tiny difference can be significant with enough data. Cohen's d measures the standardised effect size: the difference in means divided by the pooled standard deviation. Conventional thresholds are d=0.2 (small), d=0.5 (medium), d=0.8 (large). We compute Cohen's d for the most important comparisons: BC vs each PPO style, and pairwise across styles.

In [ ]:
def cohens_d(x, y):
    """Compute Cohen's d effect size."""
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx - 1) * np.std(x, ddof=1)**2 + (ny - 1) * np.std(y, ddof=1)**2) / (nx + ny - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(x) - np.mean(y)) / pooled_std

effect_rows = []
for i, m1 in enumerate(model_types):
    for m2 in model_types[i+1:]:
        rc1 = df[df["model"] == m1]["route_completion"].values
        rc2 = df[df["model"] == m2]["route_completion"].values
        if len(rc1) >= 2 and len(rc2) >= 2:
            d = cohens_d(rc1, rc2)
            magnitude = "large" if abs(d) >= 0.8 else "medium" if abs(d) >= 0.5 else "small"
            effect_rows.append({
                "Model A": m1,
                "Model B": m2,
                "Cohen's d": round(d, 4),
                "Magnitude": magnitude,
            })

df_effect = pd.DataFrame(effect_rows)
print("=== Cohen's d Effect Size (route_completion) ===")
print(df_effect.to_string(index=False))

## 4.9 Summary Tables and Figures

### 4.9.1 Master Results Table

The master results table aggregates all evaluation metrics into a single pivot table organised by model, town, and weather. Each cell shows the mean value of the metric across episodes. This table provides a comprehensive at-a-glance view of how every model performs under every condition, making it easy to identify specific weak spots (e.g., a particular model struggling in one town under rain) that warrant further investigation or targeted retraining.

In [ ]:
metrics = ["route_completion", "collision_count", "lane_keeping_frac",
           "avg_speed_kmh", "distance_m"]

for metric in metrics:
    print(f"\n=== {metric} (mean by model x town x weather) ===")
    pivot = df.pivot_table(
        values=metric,
        index=["model", "town"],
        columns="weather",
        aggfunc="mean",
    )
    print(pivot.round(3).to_string())
    print()

### 4.9.2 Export Results for Notebook 5

The evaluation results in `results/eval_results.json` are the primary input for the causal analysis in Notebook 05. Here we verify that the file is complete and report the total number of records. Each record contains the full set of per-episode metrics including model type, driving style, town, weather, route completion, collision count, lane-keeping fraction, average speed, and distance. Any missing records should be regenerated by re-running the corresponding evaluation cell above.

In [ ]:
print("=== eval_results.json Summary ===")
print(f"  File path     : {results_path}")
print(f"  File size     : {results_path.stat().st_size / 1024:.1f} KB")
print(f"  Total records : {len(df)}")
print(f"  Sensor suites : {sorted(df['sensor_suite'].unique())}")
print(f"  Models        : {sorted(df['model'].unique())}")
print(f"  Towns         : {sorted(df['town'].unique())}")
print(f"  Weathers      : {sorted(df['weather'].unique())}")

print(f"\n  Records per sensor suite:")
for suite, count in df["sensor_suite"].value_counts().items():
    print(f"    {suite}: {count}")

print("\nNotebook 04 complete. Proceed to Notebook 05 for causal analysis.")